# Federated Learning (Instructor Solution)

This version includes complete steps and a quick comparison across partitioners.

## 1) Setup

In [ ]:
!pip install -q flwr
!pip install -q "flwr-datasets[vision]"

## 2) Create the project

In [ ]:
!flwr new @flwrlabs/quickstart-pytorch

In [ ]:
%cd quickstart-pytorch
!pip install -e .

## 3) Compare partitioners

In [ ]:
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner, DirichletPartitioner, ShardPartitioner
from flwr_datasets.visualization import plot_label_distributions

def make_partitioner(kind: str, num_partitions: int = 10):
    if kind == "iid":
        return IidPartitioner(num_partitions=num_partitions)
    if kind == "dirichlet":
        return DirichletPartitioner(
            num_partitions=num_partitions,
            partition_by="fine_label",
            alpha=0.3,
            seed=42,
            min_partition_size=0,
        )
    if kind == "shard":
        return ShardPartitioner(
            num_partitions=num_partitions,
            partition_by="fine_label",
            num_shards_per_partition=2,
        )
    raise ValueError(f"Unknown partitioner: {kind}")

for kind in ["iid", "dirichlet", "shard"]:
    partitioner = make_partitioner(kind)
    fds = FederatedDataset(
        dataset="cifar100",
        partitioners={"train": partitioner},
    )
    
    print(f"--- Partitioner: {kind} ---")
    fig, ax, df = plot_label_distributions(
        partitioner,
        label_name="fine_label",
        plot_type="bar",
        size_unit="absolute",
        partition_id_axis="x",
        legend=False,
        verbose_labels=False,
        title=f"Per-Client Label Distribution ({kind})"
    )
    fig.show()

## 4) Run centralized-like baseline

Run with a single client to approximate centralized training.

In [ ]:
!flwr run . --run-config "num-server-rounds=3 learning-rate=0.05 num-clients=1"

## 5) Run federated learning

Run with multiple clients and compare the logs to the baseline.
Optionally repeat with a different number of rounds or learning rate.

In [ ]:
!flwr run . --run-config "num-server-rounds=3 learning-rate=0.05 num-clients=10"

## 6) Instructor notes

- Centralized-like (1 client) is usually faster per round and more stable.
- Federated (10 clients) can show noisier accuracy/loss and longer runtime.
- IID should look balanced; Dirichlet (alpha=0.3) should be skewed.
- Ask students to compare log trends, not just final accuracy.